# Test a bounded library assistant

Replay a library assistant that can read a public catalogue and a library schedule. Its model decisions are scripted so all learners can reproduce the same traces. No network, credentials or external actions are needed.

Fill in each starter, then run its check cell. Restart the kernel and run all cells before submitting. Keep your interpretation in the last Markdown cell.


## 1. Route a single-intent request

Route known requests to fixed paths and ambiguous requests to clarification.

Implement choose_route(intents). Return catalogue when all supplied labels are title, schedule when all supplied labels are hours, and clarify for mixed, empty or unknown labels.


In [ ]:
def choose_route(intents):
    raise NotImplementedError


In [ ]:
assert choose_route(["title","title"]) == "catalogue"
assert choose_route(["hours"]) == "schedule"
assert choose_route(["hours","title"]) == "clarify"
assert choose_route(["location"]) == "clarify"
assert choose_route([]) == "clarify"
print("OK")


## 2. Replay a read-only agent loop

Execute only preapproved read actions and stop on an answer or failure.

Implement run_reads(decisions, tools, max_reads). Consume (action, input) decisions. Return (status, answer_or_None, observations). Finish is reserved. Reject unknown actions and any read after max_reads. If decisions end without finish, return no_final_answer. Append (action, result) for each completed read.


In [ ]:
def run_reads(decisions, tools, max_reads):
    raise NotImplementedError


In [ ]:
observed = []
def lookup(value):
    observed.append(value)
    return value.upper()
tools = {"catalogue":lookup}
assert run_reads([("catalogue","map"),("finish","found")], tools, 1) == ("completed","found",[("catalogue","MAP")])
assert run_reads([("catalogue","a"),("catalogue","b")], tools, 1) == ("budget_exhausted",None,[("catalogue","A")])
assert run_reads([("unknown","a")], tools, 3) == ("unavailable_action",None,[])
assert run_reads([], tools, 0) == ("no_final_answer",None,[])
assert run_reads([("catalogue","x")], tools, 2) == ("no_final_answer",None,[("catalogue","X")]), "Keep completed observations when decisions run out"
assert observed == ["map","a","x"], "Rejected reads must never call the tool"
print("OK")


## 3. Classify a proposed read before execution

Recognize repeated proposals and exhausted budgets.

Implement read_permission(name, query, seen, used, max_reads). Return budget_exhausted first when no reads remain. Otherwise compare (name, case-folded query with whitespace collapsed) against seen and return repeated or allowed. Do not modify seen.


In [ ]:
def read_permission(name, query, seen, used, max_reads):
    raise NotImplementedError


In [ ]:
seen = {("catalogue","open map")}
assert read_permission("catalogue"," Open  MAP ",seen,1,3) == "repeated"
assert read_permission("schedule","open map",seen,1,3) == "allowed"
assert read_permission("catalogue","open map",seen,3,3) == "budget_exhausted"
assert read_permission("catalogue","new",seen,0,1) == "allowed"
assert read_permission("catalogue","Straße",{("catalogue","strasse")},0,1) == "repeated", "Use casefold, not lower"
assert seen == {("catalogue","open map")}, "Permission checks do not mutate history"
print("OK")


## 4. Report outcome counts and executed reads

Group trace events and interpret final statuses.

Implement summarize_traces(runs). Given an iterable of (terminal_status, completed_read_count) pairs, return (dictionary of counts by status, total completed reads). Include reads performed before a stopped run.


In [ ]:
def summarize_traces(runs):
    raise NotImplementedError


In [ ]:
assert summarize_traces(iter([("completed",1),("completed",0),("budget_exhausted",3)])) == ({"completed":2,"budget_exhausted":1},4)
assert summarize_traces([]) == ({},0)
assert summarize_traces([("unavailable_action",0)]) == ({"unavailable_action":1},0)
print("OK")


## 5. Add a repeat guard to an agent replay

Implement guarded_reads(decisions, tools, max_reads). Keep run_reads result and status semantics,
with one new status: repeated. Recognize finish and unknown actions first, then call
read_permission before dispatching. Remember executed (name, normalized query) keys. A blocked
action adds no observation and must not execute. The read budget has priority over repetition.


In [ ]:
def guarded_reads(decisions, tools, max_reads):
    raise NotImplementedError


In [ ]:
seen_calls = []
def catalogue(query):
    seen_calls.append(query)
    return "North branch"
guard_tools = {"catalogue":catalogue, "schedule":lambda branch:"09:00"}
multi_step = guarded_reads([("catalogue","maps"),("schedule","North branch"),("finish","North opens at 09:00")], guard_tools, 2)
repeat = guarded_reads([("catalogue"," Maps "),("catalogue","maps")], guard_tools, 3)
cap = guarded_reads([("catalogue","maps"),("catalogue","maps")], guard_tools, 1)
assert multi_step == ("completed","North opens at 09:00",[("catalogue","North branch"),("schedule","09:00")])
assert repeat == ("repeated",None,[("catalogue","North branch")])
assert cap == ("budget_exhausted",None,[("catalogue","North branch")])
assert seen_calls == ["maps"," Maps ","maps"], "Each run does one catalogue read before it stops"
assert guarded_reads([("unknown","x")], guard_tools, 0) == ("unavailable_action",None,[]), "Recognize unknown actions before the budget check"
assert summarize_traces((status, len(observations)) for status, _, observations in [multi_step, repeat, cap]) == ({"completed":1,"repeated":1,"budget_exhausted":1},4)
print("OK")


## Interpretation

Compare a fixed route and a bounded loop for the same library question. Identify what genuinely requires a second lookup and which stop status should be shown if the second lookup repeats the first proposal.

**Your answer**
